In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

cwd = os.getcwd()
root_path = os.path.abspath(os.path.join(cwd, os.pardir))
sys.path.append(root_path)

In [ ]:
from implementation.regression_methods_comparison import compare_methods, rank_methods

np.random.seed(42)
x_data = np.linspace(0, 5, 20)
y_data = np.sin(x_data) + np.random.normal(0, 0.1, size=len(x_data))


def poly_fit_1(x, y):
    coeffs = np.polyfit(x, y, 1)
    return np.polyval(coeffs, x)


def poly_fit_2(x, y):
    coeffs = np.polyfit(x, y, 2)
    return np.polyval(coeffs, x)


def poly_fit_5(x, y):
    coeffs = np.polyfit(x, y, 5)
    return np.polyval(coeffs, x)


methods = {
    "Linear (deg 1)": poly_fit_1,
    "Quadratic (deg 2)": poly_fit_2,
    "Degree 5": poly_fit_5,
}

results = compare_methods(x_data, y_data, methods)
ranking = rank_methods(results, metric="rmse")

print("Method Comparison Results:")
for name, metrics in results.items():
    print(f"\n  {name}:")
    for k, v in metrics.items():
        print(f"    {k}: {v:.6f}")

print("\nRanking by RMSE (lower is better):")
for rank, (name, score) in enumerate(ranking, 1):
    print(f"  {rank}. {name}: {score:.6f}")

In [ ]:
x_fine = np.linspace(0, 5, 200)
degrees = {"Linear (deg 1)": 1, "Quadratic (deg 2)": 2, "Degree 5": 5}

plt.figure(figsize=(12, 6))
plt.scatter(x_data, y_data, color="black", s=60, zorder=5, label="Data")

colors = ["blue", "green", "orange"]
for (name, method), color in zip(methods.items(), colors):
    coeffs = np.polyfit(x_data, y_data, degrees[name])
    y_fine = np.polyval(coeffs, x_fine)
    rmse = results[name]["rmse"]
    plt.plot(x_fine, y_fine, color=color, label=f"{name} (RMSE={rmse:.4f})")

plt.plot(x_fine, np.sin(x_fine), "k--", alpha=0.3, label="sin(x)")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Regression Methods Comparison")
plt.legend()
plt.grid(True)
plt.show()

## Held-out ranking (cross-validation)

`compare_methods` scores each model on the points it was fitted to, so the
most flexible model tends to win. `cross_validate_methods` predicts every point
with a model that did not see it; the pooled held-out errors can be ranked
with the same `rank_methods`.

In [ ]:
from implementation.regression_methods_comparison import (
    cross_validate_methods,
    polynomial_fit_predict,
)

cv_methods = {name: polynomial_fit_predict(d) for name, d in degrees.items()}
cv_results = cross_validate_methods(x_data, y_data, cv_methods, n_folds=5, seed=0)
cv_ranking = rank_methods(cv_results, metric="rmse")

print("Ranking by cross-validated RMSE (lower is better):")
for rank, (name, score) in enumerate(cv_ranking, 1):
    print(f"  {rank}. {name}: {score:.6f}  (training RMSE {results[name]['rmse']:.6f})")